# MP-Declare Constraint Mining

Mining MP-Declare constraints with data conditions from the BPIC17 event log using RuM's MINERful + MpEnhancer.

In [ ]:
import sys
import os
from pathlib import Path

os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home'

_current = Path().resolve()
while _current != _current.parent:
    if (_current / 'src').is_dir():
        break
    _current = _current.parent

if str(_current) not in sys.path:
    sys.path.insert(0, str(_current))

from src.interpretability.perturbation_methods import csv_to_xes, discover_mpdeclare

In [ ]:
csv_path = _current / 'data' / 'BPI_Challenge_2017.csv'
xes_path = _current / 'data' / 'BPI_Challenge_2017.xes'

if not xes_path.exists():
    csv_to_xes(csv_path, xes_path, case_id_col="case:concept:name", activity_col="concept:name", timestamp_col="time:timestamp")
    print("Converted CSV to XES")
else:
    print(f"XES file already exists: {xes_path}")

In [ ]:
constraints = discover_mpdeclare(xes_path, min_support=0.99, data_conditions='ACTIVATIONS')
print(f"Mined {len(constraints)} MP-Declare constraints")

In [ ]:
print("=" * 100)
print("MP-DECLARE CONSTRAINTS WITH DATA CONDITIONS")
print("=" * 100)
print()

with_data = [c for c in constraints if c.data_condition]
print(f"Found {len(with_data)} constraints with data conditions:\n")

for i, c in enumerate(with_data, 1):
    print(f"{i}. {c.template}[{c.activation}, {c.target}]")
    print(f"   Support: {c.support:.1%}")
    print(f"   Data condition: {c.data_condition}")
    print()

In [ ]:
print("=" * 100)
print("ALL MINED CONSTRAINTS")
print("=" * 100)
print()

for i, c in enumerate(constraints, 1):
    print(f"{i:3}. {c}")

# REVISED+ Counterfactual Explanations

Generate counterfactual prefixes using the REVISED+ orchestrator:
- VAE trained on **random-length prefixes** (learns the prefix manifold)
- **Two-tier plausibility**: prefix-safe constraints for search penalty, all constraints for validity gate
- Latent space elite-sampling search

**Note**: BPIC17 is significantly larger than Helpdesk/DomesticDeclarations (~6300 test cases, 9 categorical + 9 numerical features). VAE training will take longer on first run.

In [ ]:
import torch

import sys
src_path = str(_current / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# --- Load dataset ---
data_path = _current / 'encoded_data' / 'BPIC_2017_all_5_test.pkl'
dataset = torch.load(data_path, weights_only=False)

sample = dataset[0]
n_cat = len(sample[0])
n_num = len(sample[1])
seq_len = sample[0][0].shape[0]
print(f"Dataset: {len(dataset)} sequences, {n_cat} cat, {n_num} num, seq_len={seq_len}")

# --- Load trained prediction model ---
from src.model.dropout_uncertainty_enc_dec_LSTM.dropout_uncertainty_model import DropoutUncertaintyEncoderDecoderLSTM

model_path = _current / 'src' / 'notebooks' / 'training_variational_dropout' / 'BPIC17' / 'BPIC_2017_full_grad_norm_new_4layer.pkl'
model = DropoutUncertaintyEncoderDecoderLSTM.load(str(model_path), dropout=0.0)
model.eval()
print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")

In [ ]:
from src.interpretability.utils.tensor_decoder import TensorDecoder

decoder = TensorDecoder(dataset)

# Build activity_names list: index -> name (for the REVISED+ orchestrator)
# BPIC17 uses 'concept:name' as the activity feature (not 'Activity')
ACTIVITY_FEATURE = 'concept:name'

activity_idx_to_label = decoder.idx_to_label[ACTIVITY_FEATURE]
max_idx = max(activity_idx_to_label.keys())
activity_names = [activity_idx_to_label.get(i, f'<unk_{i}>') for i in range(max_idx + 1)]

# Find EOS index
eos_idx = next(i for i, name in enumerate(activity_names) if name == 'EOS')

print(f"Activity vocabulary ({len(activity_names)}):")
for i, name in enumerate(activity_names):
    marker = ' <-- EOS' if i == eos_idx else ''
    print(f"  {i}: {name}{marker}")

In [ ]:
from src.interpretability.perturbation_methods import RevisedPlus, RevisedPlusConfig, create_revised_plus_for_model

device = 'mps' if torch.backends.mps.is_available() else 'cpu'

config = RevisedPlusConfig(
    vae_epochs=100,
    vae_kl_weight=0.1,
    declare_min_support=0.9,
    n_candidates_per_round=200,
    n_search_rounds=5,
    top_k=5,
    min_plausibility=0.0,
    device=device,
)

vae_path = str(_current / 'encoded_data' / 'bpic17_vae.pkl')

rp = create_revised_plus_for_model(
    model=model,
    dataset=dataset,
    activity_names=activity_names,
    config=config,
    vae_path=vae_path,
)
print(f"\nREVISED+ ready:")
print(f"  VAE parameters: {sum(p.numel() for p in rp.vae.parameters()):,}")
print(f"  VAE path: {vae_path}")
print(f"  All constraints: {len(rp.all_constraints)}")
print(f"  Prefix-safe constraints: {len(rp.prefix_safe_constraints)}")

In [ ]:
print("PREFIX-SAFE constraints (used in search penalty):")
print("-" * 60)
for c in sorted(rp.prefix_safe_constraints, key=lambda x: (x.template.value, x.activities)):
    print(f"  {c.format(activity_names)}")

print(f"\nNOT prefix-safe constraints (used in validity gate):")
print("-" * 60)
not_safe = rp.all_constraints - rp.prefix_safe_constraints
for c in sorted(not_safe, key=lambda x: (x.template.value, x.activities)):
    print(f"  {c.format(activity_names)}")

In [ ]:
import numpy as np

# Find a good candidate: in-progress prefix with uncertain prediction
best_idx, best_prob = None, 1.0
for i in range(min(200, len(dataset))):
    cat_t, num_t, _ = dataset[i]
    act = cat_t[0]  # concept:name is the first categorical feature
    # Skip completed traces (contain EOS)
    if (act == eos_idx).any():
        continue
    cat_in = [c.unsqueeze(0) for c in cat_t]
    num_in = [n.unsqueeze(0) for n in num_t]
    with torch.no_grad():
        preds = model((cat_in, num_in))[0]
        logits = preds[0][f"{ACTIVITY_FEATURE}_mean"][0]
        p = torch.softmax(logits, dim=-1)
        top_p = p.max().item()
    if 0.4 < top_p < best_prob:
        best_idx, best_prob = i, top_p

test_idx = best_idx if best_idx is not None else 42
print(f"Selected test_idx={test_idx} (top_p={best_prob:.3f})")

cat_tuple, num_tuple, case_id = dataset[test_idx]

print(f"\nCase: {case_id}")
df_orig = decoder.decode_sequence(cat_tuple, num_tuple, case_id=case_id)
display(df_orig)

cat_tensors = [c.unsqueeze(0) for c in cat_tuple]
num_tensors = [n.unsqueeze(0) for n in num_tuple]

explanation = rp.explain(cat_tensors, num_tensors, target_class=None)
print(explanation)

In [ ]:
if explanation.counterfactuals:
    best = explanation.get_best()

    print("=" * 80)
    print("ORIGINAL PREFIX")
    print(f"Prediction: {explanation.original_prediction_name} (p={explanation.original_probability:.3f})")
    print(f"Prefix length: {explanation.prefix_len} events")
    print("=" * 80)
    display(df_orig)

    print(f"\n{'=' * 80}")
    print("BEST COUNTERFACTUAL PREFIX")
    print(f"Prediction: {best.counterfactual_prediction_name} (p={best.counterfactual_probability:.3f})")
    cf_prefix_len = len(best.activity_sequence)
    print(f"Prefix length: {cf_prefix_len} events (delta={cf_prefix_len - explanation.prefix_len:+d})")
    print(f"Proximity: {best.proximity:.3f}  Sparsity: {best.sparsity}")
    print(f"Feasibility: {best.feasibility:.3f}")
    print(f"Plausibility (definite): {best.plausibility_definite:.2f}")
    print(f"Plausibility (optimistic): {best.plausibility_optimistic:.2f}")
    print(f"Combined score: {best.combined_score:.4f}")
    print("=" * 80)

    cf_cat = tuple(best.cat_sequence)
    cf_num = tuple(best.num_sequence[:, i] for i in range(best.num_sequence.shape[1])) if best.num_sequence is not None else num_tuple
    df_cf = decoder.decode_sequence(cf_cat, cf_num, skip_padding=False)
    display(df_cf)
else:
    print("No counterfactuals found. Try increasing n_search_rounds or noise_scale.")

In [ ]:
import pandas as pd

if explanation.counterfactuals:
    rows = []
    for i, cf in enumerate(explanation.counterfactuals):
        rows.append({
            'rank': i + 1,
            'prediction': cf.counterfactual_prediction_name,
            'probability': f"{cf.counterfactual_probability:.3f}",
            'prefix_len': len(cf.activity_sequence),
            'activities': ' -> '.join(activity_names[a] for a in cf.activity_sequence),
            'proximity': f"{cf.proximity:.2f}",
            'sparsity': cf.sparsity,
            'feasibility': f"{cf.feasibility:.3f}",
            'plaus_def': f"{cf.plausibility_definite:.2f}",
            'plaus_opt': f"{cf.plausibility_optimistic:.2f}",
            'score': f"{cf.combined_score:.4f}",
        })

    df_cfs = pd.DataFrame(rows)
    print(f"Original: {' -> '.join(activity_names[a] for a in explanation.original_activity_sequence)}")
    print(f"Original prediction: {explanation.original_prediction_name} (p={explanation.original_probability:.3f})")
    print()
    display(df_cfs)